In [ ]:
# Install dependencies
!pip install -q pandas scikit-learn joblib tqdm datasets transformers torch

In [ ]:
import os, zipfile, requests, sys
url = "https://github.com/vedangvatsa/ai-detection-at-scale/archive/refs/heads/main.zip"
r = requests.get(url)
open("repo.zip", "wb").write(r.content)
with zipfile.ZipFile("repo.zip") as z:
    z.extractall(".")
os.rename("ai-detection-at-scale-main", "ai-detection-at-scale")
os.chdir("ai-detection-at-scale")
sys.path.insert(0, '.')
print("Repository structure loaded successfully.")

In [ ]:
# Download local assets (GPT-2 Small + Stylometric models)
!python scripts/download_assets.py

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Load Beemo dataset
print("Loading Toloka Beemo dataset...")
beemo = load_dataset("toloka/beemo")
df = beemo['train'].to_pandas()

# Split into Train and Test indices (80% / 20%)
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)
train_df = df.loc[train_idx]
test_df = df.loc[test_idx]

def extract_pairs(dataframe):
    records = []
    for _, row in dataframe.iterrows():
        h = row['human_output']
        m = row['model_output']
        if isinstance(h, str) and len(h.strip()) >= 20:
            records.append({'text': h, 'label': 0})
        if isinstance(m, str) and len(m.strip()) >= 20:
            records.append({'text': m, 'label': 1})
    return pd.DataFrame(records)

train_data = extract_pairs(train_df)
test_data = extract_pairs(test_df)
print(f"Train pairs: {len(train_data)}, Test pairs: {len(test_data)}")

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

MODEL_NAME = "prajjwal1/bert-mini"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
        
    def __len__(self):
        return len(self.labels)

# Take a balanced subset of training data to train efficiently under 5 minutes
train_subset = train_data.sample(n=min(3000, len(train_data)), random_state=42).reset_index(drop=True)
train_dataset = TextDataset(train_subset['text'].tolist(), train_subset['label'].tolist(), tokenizer)

device = "cpu"
print(f"Using device: {device}")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    learning_rate=3e-5,
    fp16=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

print("Training semantic transformer on Beemo MGT targets...")
trainer.train()
print("Transformer fine-tuning completed.")
model.save_pretrained("beemo_semantic_model")
tokenizer.save_pretrained("beemo_semantic_model")

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from tool.feature_extractor import extract_features
from tool.neural_detector import compute_perplexity_and_burstiness
from tqdm import tqdm

model.eval()
def get_semantic_prob(text):
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256, padding=True).to(device)
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)
        return probs[0][1].item()

# Generate joint features on a validation subset of Beemo
val_sample = test_data.sample(n=min(2000, len(test_data)), random_state=42).reset_index(drop=True)

X_ensemble = []
y_ensemble = []

print("Extracting hybrid ensembled features (Stylometrics + Neural PPL + Transformer Probability)...")
for _, row in tqdm(val_sample.iterrows(), total=len(val_sample)):
    text = row['text']
    label = row['label']
    
    # 1. Stylometrics (11 features)
    feats = extract_features(text, extended=False)
    if feats is None:
        continue
    stylo_vector = [feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                      'connector_density', 'hedge_density', 'mean_sent_len',
                                      'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']]
    
    # 2. Neural PPL & Burstiness (GPT-2 Small)
    try:
        ppl, burst = compute_perplexity_and_burstiness(text)
    except:
        ppl, burst = 50.0, 1.0
        
    # 3. Transformer Prob
    sem_prob = get_semantic_prob(text)
    
    # Combine all into feature vector
    vec = stylo_vector + [ppl, burst, sem_prob]
    X_ensemble.append(vec)
    y_ensemble.append(label)

X_ensemble = np.array(X_ensemble)
y_ensemble = np.array(y_ensemble)

# Train a calibrated ensembling logistic regression classifier
ensembler = LogisticRegression(max_iter=1000)
ensembler.fit(X_ensemble, y_ensemble)

# Print training metrics
train_preds = ensembler.predict_proba(X_ensemble)[:, 1]
train_auc = roc_auc_score(y_ensemble, train_preds)
print(f"Ensembler Train AUC on Beemo subset: {train_auc:.4f}")

# Save ensembler checkpoint
joblib.dump(ensembler, "beemo_hybrid_ensembler.joblib")

In [ ]:
# Evaluate on a held-out test split of Beemo to confirm SOTA generalization
test_sample_eval = test_data.exclude(val_sample.index) if hasattr(test_data, 'exclude') else test_data.iloc[~test_data.index.isin(val_sample.index)]
test_sample_eval = test_sample_eval.sample(n=min(1000, len(test_sample_eval)), random_state=100).reset_index(drop=True)

X_test = []
y_test = []

print("Evaluating hybrid ensembler on held-out Beemo test cases...")
for _, row in tqdm(test_sample_eval.iterrows(), total=len(test_sample_eval)):
    text = row['text']
    label = row['label']
    
    feats = extract_features(text, extended=False)
    if feats is None:
        continue
    stylo_vector = [feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                      'connector_density', 'hedge_density', 'mean_sent_len',
                                      'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']]
    try:
        ppl, burst = compute_perplexity_and_burstiness(text)
    except:
        ppl, burst = 50.0, 1.0
    sem_prob = get_semantic_prob(text)
    
    X_test.append(stylo_vector + [ppl, burst, sem_prob])
    y_test.append(label)

X_test = np.array(X_test)
y_test = np.array(y_test)

test_preds = ensembler.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_preds)
print(f"--- SOTA BEEMO TEST AUC: {test_auc:.4f} ---")

In [ ]:
# Zip checkpoints for download
import shutil
shutil.make_archive("/kaggle/working/beemo_semantic_model", 'zip', "beemo_semantic_model")
shutil.copy("beemo_hybrid_ensembler.joblib", "/kaggle/working/beemo_hybrid_ensembler.joblib")
print("Checkpoints archived and ready in /kaggle/working/")